In [ ]:
# ==========================================
# 8-MODEL WBF ENSEMBLE (ZERO-TRAINING)
# ==========================================
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Install dependencies if missing
try:
    from ultralytics import YOLO
except ImportError:
    import subprocess; subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)
    from ultralytics import YOLO
try:
    from ensemble_boxes import weighted_boxes_fusion
except ImportError:
    import subprocess; subprocess.run(["pip", "install", "-q", "ensemble-boxes"], check=True)
    from ensemble_boxes import weighted_boxes_fusion

# ==========================================
# 1. PATH RESOLUTION
# ==========================================
INPUT_ROOT = Path("/kaggle/input")

# Your exact provided paths for the 3-fold and 5-fold weights
DIR_3FOLD = Path("/kaggle/input/notebooks/tahjibhossainkhan/k-fold-version-2")
DIR_5FOLD = Path("/kaggle/input/notebooks/tahjibhossainkhan/final-k-fold-folds-5")

# Locate Test Images dynamically (requires the competition dataset to be attached)
image_dirs = [p for p in INPUT_ROOT.rglob("images") if p.is_dir()]
TEST_IMAGES  = next((p for p in image_dirs if "test" in str(p).lower()), None)

print(f"Test Images located at: {TEST_IMAGES}")
assert TEST_IMAGES is not None, "Test images not found! Make sure the competition dataset is attached via 'Add Input'."

# ==========================================
# 2. LOCATE & LOAD WEIGHTS
# ==========================================
# We use rglob to hunt down all 'best.pt' files in your attached notebooks
weights_3f = list(DIR_3FOLD.rglob("best.pt"))
weights_5f = list(DIR_5FOLD.rglob("best.pt"))

print(f"Found {len(weights_3f)} weights from the 3-fold run.")
print(f"Found {len(weights_5f)} weights from the 5-fold run.")

assert len(weights_3f) == 3, f"Could not find exactly 3 weights in {DIR_3FOLD}"
assert len(weights_5f) == 5, f"Could not find exactly 5 weights in {DIR_5FOLD}"

all_weights = weights_3f + weights_5f
print("\nLoading all 8 YOLO11m models into memory...")
models = [YOLO(str(w)) for w in all_weights]

# ==========================================
# 3. INFERENCE CONFIGURATION
# ==========================================
IMGSZ = 1280
USE_TTA = True               
PRED_CONF = 0.001            # Maximum recall baseline
PRED_IOU = 0.60              # Stable IoU for overlapping suppression
WBF_IOU = 0.60               # WBF fusion threshold
SKIP_THR = 0.0001            # Keep low-confidence boxes before fusion

# ==========================================
# 4. ENSEMBLE EXECUTION
# ==========================================
test_idx = {p.stem: p for p in TEST_IMAGES.rglob("*.*") if p.suffix.lower() in [".jpg", ".png", ".jpeg"]}
test_stems = sorted(test_idx.keys())
print(f"\nCommencing inference on {len(test_stems)} images...")

rows = []
for stem in test_stems:
    p = str(test_idx[stem])
    boxes_l, scores_l, labels_l = [], [], []
    
    # Query all 8 models for this specific image
    for m in models:
        b = m.predict(source=p, imgsz=IMGSZ, conf=PRED_CONF, iou=PRED_IOU,
                      max_det=300, augment=USE_TTA, verbose=False)[0].boxes
        
        if b is not None and len(b) > 0:
            boxes_l.append(b.xyxyn.cpu().numpy().tolist())
            scores_l.append(b.conf.cpu().numpy().tolist())
            labels_l.append(b.cls.cpu().numpy().astype(int).tolist())

    if not boxes_l:
        rows.append({"image_id": stem, "PredictionString": ""})
        continue

    # Mathematically fuse the 8 sets of predictions
    fb, fs, fl = weighted_boxes_fusion(boxes_l, scores_l, labels_l,
                                       iou_thr=WBF_IOU, skip_box_thr=SKIP_THR)
    parts = []
    for i in np.argsort(fs)[::-1]:
        x1, y1, x2, y2 = fb[i]
        # Kaggle submission format mapping
        parts += [str(int(fl[i])), f"{fs[i]*100:.2f}",
                  f"{(x1+x2)/2:.6f}", f"{(y1+y2)/2:.6f}", f"{x2-x1:.6f}", f"{y2-y1:.6f}"]
                  
    rows.append({"image_id": stem, "PredictionString": " ".join(parts)})

# ==========================================
# 5. PACKAGING 
# ==========================================
sub = pd.DataFrame(rows).sort_values("image_id").reset_index(drop=True)
SUB_PATH = "/kaggle/working/submission_ensemble8.csv"
sub.to_csv(SUB_PATH, index=False)

n_with = int((sub["PredictionString"].str.len() > 0).sum())
print(f"\nSUCCESS! Saved {SUB_PATH}")
print(f"Rows: {len(sub)} (Expected: {len(test_stems)}) | Images with detections: {n_with}")
assert len(sub) == len(test_stems), "CRITICAL: Row count mismatch!"